# Verify `virchow2_backbone_best.pt` against the original backbone

This notebook checks whether `runs0708/virchow2_backbone_best.pt` matches the original `virchow2` backbone everywhere except the final two transformer blocks.

Note: the training code in `fine-tune.py` also leaves the final `norm.*` parameters trainable, so this notebook treats `blocks.30.*`, `blocks.31.*`, and `norm.*` as the only places where differences are allowed.

In [2]:
from collections import Counter
from pathlib import Path

import torch

from helpers import get_model_and_transform

In [3]:
checkpoint_path = Path('runs0708/virchow2_backbone_best.pt')
assert checkpoint_path.exists(), f'Missing checkpoint: {checkpoint_path}'

backbone, _ = get_model_and_transform('virchow2')
original_state = backbone.model.state_dict()
finetuned_state = torch.load(checkpoint_path, map_location='cpu', weights_only=True)

print(f'Original parameter tensors: {len(original_state)}')
print(f'Checkpoint parameter tensors: {len(finetuned_state)}')
print(f'Keys identical: {set(original_state) == set(finetuned_state)}')

Original parameter tensors: 455
Checkpoint parameter tensors: 455
Keys identical: True


In [9]:
original_keys = list(original_state.keys())
num_blocks = len(backbone.model.blocks)
expected_prefixes = [
    f'blocks.{num_blocks - 2}.',
    f'blocks.{num_blocks - 1}.',
    'norm.',
]

changed_keys = [
    key
    for key in original_keys
    if not torch.equal(original_state[key], finetuned_state[key])
]
unchanged_keys = [key for key in original_keys if key not in changed_keys]

unexpected_changed = [
    key for key in changed_keys
    if not any(key.startswith(prefix) for prefix in expected_prefixes)
]
expected_region_changes = [
    key for key in changed_keys
    if any(key.startswith(prefix) for prefix in expected_prefixes)
]

print('Expected diff prefixes:')
for prefix in expected_prefixes:
    print('  ', prefix)

print(f'Changed tensors: {len(changed_keys)}')
print(f'Unchanged tensors: {len(unchanged_keys)}')
print(f'Changed only in allowed regions: {len(unexpected_changed) == 0}')
print(f'Changed tensors inside allowed regions: {len(expected_region_changes)}')

Expected diff prefixes:
   blocks.30.
   blocks.31.
   norm.
Changed tensors: 30
Unchanged tensors: 425
Changed only in allowed regions: True
Changed tensors inside allowed regions: 30


In [5]:
def summarize(keys):
    summary = Counter()
    for key in keys:
        parts = key.split('.')
        if parts[0] == 'blocks' and len(parts) > 1:
            summary[f'{parts[0]}.{parts[1]}'] += 1
        else:
            summary[parts[0]] += 1
    return summary

print('Changed tensors by module:')
for module_name, count in summarize(changed_keys).most_common():
    print(f'  {module_name}: {count}')

if unexpected_changed:
    print('\nUnexpected changed tensors:')
    for key in unexpected_changed:
        print('  ', key)
else:
    print('\nNo unexpected changed tensors found.')

Changed tensors by module:
  blocks.30: 14
  blocks.31: 14
  norm: 2

No unexpected changed tensors found.


In [6]:
assert set(original_state) == set(finetuned_state), 'Checkpoint keys do not match the original backbone.'
assert expected_region_changes, 'No differences found in the allowed fine-tuned regions.'
assert not unexpected_changed, (
    'Found weight changes outside the final two transformer blocks and final norm: '
    + ', '.join(unexpected_changed[:10])
)

print('Verification passed: only the final two transformer blocks and final norm differ from the original backbone.')

Verification passed: only the final two transformer blocks and final norm differ from the original backbone.
